# 2. 데이터 전처리

결측치·이상치를 처리하고, 피처 엔지니어링·스케일링·클러스터링을 적용하여 모델 학습에 적합한 데이터를 만듭니다.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

## 2.1 결측치 처리

수치형 변수의 결측치를 각 데이터셋의 평균으로 대체합니다.

In [ ]:
def fill_na_with_mean(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mean())
    return df

train = fill_na_with_mean(train)
test = fill_na_with_mean(test)
print(f"처리 후 결측치 - Train: {train.isnull().sum().sum()}, Test: {test.isnull().sum().sum()}")

## 2.2 이상치 처리

의학적 정상 범위와 IQR 기반 클리핑을 적용하여 극단값을 제한합니다.

In [ ]:
def preprocess_outliers(df):
    df = df.copy()

    def clip_iqr(series):
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        return series.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    df['키(cm)'] = clip_iqr(df['키(cm)'])
    df['시력'] = df['시력'].clip(0.1, 2)
    df['공복 혈당'] = df['공복 혈당'].clip(50, 300)
    df['혈압'] = df['혈압'].clip(10, 150)
    df['중성 지방'] = df['중성 지방'].clip(0, 500)
    df['혈청 크레아티닌'] = df['혈청 크레아티닌'].clip(0, 5)
    df['콜레스테롤'] = df['콜레스테롤'].clip(0, 400)
    df['고밀도지단백'] = df['고밀도지단백'].clip(20, 160)
    df['저밀도지단백'] = df['저밀도지단백'].clip(0, 250)
    return df

clean_train = preprocess_outliers(train)
clean_test = preprocess_outliers(test)
print(f"이상치 처리 후 - Train: {clean_train.shape}, Test: {clean_test.shape}")

## 2.3 피처 엔지니어링

도메인 지식(의학적 지표)을 활용하여 파생 변수를 생성합니다.
- 지질 관련 비율 (동맥경화지수, 심혈관위험도 등)
- 비만도 관련 지표 (BMI 재계산, 복부비만지수)
- 대사증후군 위험 점수
- 연령대 구간화 및 교호작용 변수

In [ ]:
def add_features(df):
    df = df.copy()
    eps = 1e-5

    # 연령대 구간화
    df['연령대'] = pd.cut(df['나이'], bins=[0, 30, 40, 50, 60, 70, 100], labels=[0, 1, 2, 3, 4, 5]).astype(int)

    # 비만도 관련
    height_m = df['키(cm)'] / 100
    df['계산된BMI'] = df['몸무게(kg)'] / (height_m ** 2)
    df['BMI차이'] = df['BMI'] - df['계산된BMI']
    df['BMI제곱'] = df['계산된BMI'] ** 2
    df['BMI_group'] = pd.cut(df['BMI'], bins=[0, 23, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)
    df['age_BMI_interact'] = df['BMI_group'] * df['연령대']
    df['비만여부'] = (df['계산된BMI'] >= 30).astype(int)
    df['과체중여부'] = (df['계산된BMI'] >= 25).astype(int)

    # 지질 비율 및 로그 변환
    df['동맥경화지수'] = np.log10((df['중성 지방'] + eps) / df['고밀도지단백'])
    df['심혈관위험도1'] = df['콜레스테롤'] / df['고밀도지단백']
    df['심혈관위험도2'] = df['저밀도지단백'] / df['고밀도지단백']
    df['중성지방_HDL비율'] = df['중성 지방'] / df['고밀도지단백']
    df['비HDL콜레스테롤'] = df['콜레스테롤'] - df['고밀도지단백']
    df['잔여콜레스테롤'] = df['중성 지방'] / 5
    df['중성지방_LDL비율'] = df['중성 지방'] / (df['저밀도지단백'] + eps)
    df['HDL비율'] = df['고밀도지단백'] / df['콜레스테롤']
    df['복부비만지수1'] = df['계산된BMI'] * df['중성 지방'] / (df['고밀도지단백'] + eps)
    df['복부비만지수2'] = df['계산된BMI'] * df['동맥경화지수']

    # 위험군 플래그
    df['콜레스테롤위험군'] = ((df['콜레스테롤'] >= 240) | (df['저밀도지단백'] >= 160)).astype(int)
    df['중성지방고위험군'] = (df['중성 지방'] >= 200).astype(int)

    # 대사증후군 점수
    df['대사_혈당위험'] = (df['공복 혈당'] >= 100).astype(int)
    df['대사_중성지방위험'] = (df['중성 지방'] >= 150).astype(int)
    df['대사_HDL위험'] = (df['고밀도지단백'] < 40).astype(int)
    df['대사증후군점수'] = df['대사_혈당위험'] + df['대사_중성지방위험'] + df['대사_HDL위험']

    # 교호작용 변수
    df['공복혈당_시력'] = df['공복 혈당'] * df['시력']
    df['헤모글로빈_나이'] = df['헤모글로빈'] * df['나이']
    df['헤모글로빈_혈압'] = df['헤모글로빈'] * df['혈압']
    df['중성지방_헤모글로빈'] = df['중성 지방'] * df['헤모글로빈']

    return df

train_featured = add_features(clean_train)
test_featured = add_features(clean_test)
print(f"피처 엔지니어링 후 - Train: {train_featured.shape}, Test: {test_featured.shape}")

## 2.4 데이터 분리 및 스케일링 + 클러스터링

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

# ID 저장 및 데이터 분리
train_ids = train_featured['ID']
test_ids = test_featured['ID']
train_raw = train_featured.drop(['ID', 'label'], axis=1)
test_raw = test_featured.drop(['ID'], axis=1)
train_target = train_featured['label']

# KMeans 클러스터링 피처 추가
scaler_temp = RobustScaler()
X_temp = scaler_temp.fit_transform(train_raw)

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
train_raw['cluster'] = kmeans.fit_predict(X_temp)
test_raw['cluster'] = kmeans.predict(scaler_temp.transform(test_raw))

# 학습용 데이터 결합
train_data = pd.concat([train_raw, train_target], axis=1)
print(f"최종 학습 데이터: {train_data.shape}")
print(f"최종 테스트 데이터: {test_raw.shape}")

## 2.5 전처리된 데이터 저장

In [ ]:
train_data.to_csv('../data/train_processed.csv', index=False)
test_raw.to_csv('../data/test_processed.csv', index=False)
test_ids.to_csv('../data/test_ids.csv', index=False)
print("전처리된 데이터 저장 완료")